# LLM Query Index
Query an Azure AI Search index using Azure OpenAI with the "on your data" extension. Run **cells 1–4 once** to initialize, then re-run **cell 5** repeatedly to test different queries.

## 1. Import Required Libraries

In [1]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

## 2. Configuration and Setup
Load environment variables and define all configuration values. **Run once.**

In [23]:
load_dotenv()

endpoint = os.getenv("ENDPOINT_URL", "https://poc-sweden-ai-plattform.openai.azure.com/")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
search_endpoint = os.getenv("SEARCH_ENDPOINT", "https://pocaisearchbam.search.windows.net")
search_key = os.environ["AI_SEARCH_KEY"]
search_index = "push-semantic-chunking-1"
subscription_key = os.environ["AZURE_OPENAI_API_KEY"]
embedding_deployment = os.getenv("EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-large")

print(f"Endpoint:            {endpoint}")
print(f"Deployment:          {deployment}")
print(f"Embedding deployment:{embedding_deployment}")
print(f"Search endpoint:     {search_endpoint}")
print(f"Search index:        {search_index}")

Endpoint:            https://poc-sweden-ai-plattform.openai.azure.com/
Deployment:          gpt-4.1
Embedding deployment:text-embedding-3-large
Search endpoint:     https://pocaisearchbam.search.windows.net
Search index:        push-semantic-chunking-1


## 3. Initialize Azure OpenAI Client
Create the client and define the Azure AI Search data source configuration. **Run once.**

In [26]:
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)

data_sources = [{
    "type": "azure_search",
    "parameters": {
        "endpoint": search_endpoint,
        "index_name": search_index,
        "semantic_configuration": "my-semantic-config",
        "query_type": "vector_semantic_hybrid",
        "fields_mapping": {
            "title_field": "section_heading",
            "content_fields": ["chunk"],
            "vector_fields": ["chunkVector"],
        },
        "in_scope": True,
        "filter": None,
        "strictness": 3,
        "top_n_documents": 5,
        "include_contexts": ["citations", "all_retrieved_documents"],
        "embedding_dependency": {
            "type": "deployment_name",
            "deployment_name": embedding_deployment,
        },
        "authentication": {
            "type": "api_key",
            "key": search_key,
        },
    },
}]

print("Client and data source configuration ready.")

Client and data source configuration ready.


## 4. Query Parameters
Tune completion parameters. **Run once**, or adjust and re-run before a new query session.

In [21]:
system_message = "You are an AI assistant that helps people find information."

# Completion parameters
max_tokens = 13107
temperature = 0.7
top_p = 0.95
frequency_penalty = 0
presence_penalty = 0

## 5. Run Query
Edit `user_query` and re-run this cell as many times as needed to test different questions.

In [27]:
from IPython.display import display, Markdown

# ── Edit this to test different questions ──────────────────────────────────────
user_query = "Welche Tabelle enthält die Maximaltemperaturen für Routine Beförderungen?"
# ───────────────────────────────────────────────────────────────────────────────

messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": user_query},
]

completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    max_tokens=max_tokens,
    temperature=temperature,
    top_p=top_p,
    frequency_penalty=frequency_penalty,
    presence_penalty=presence_penalty,
    stop=None,
    stream=False,
    extra_body={"data_sources": data_sources},
)

# Build score lookup: title -> (search_score, rerank_score)
ctx = completion.choices[0].message.context
score_map = {
    doc["title"]: (doc.get("original_search_score"), doc.get("rerank_score"))
    for doc in ctx.get("all_retrieved_documents", [])
}

# Display the answer and citations
answer = completion.choices[0].message.content
citations = ctx.get("citations", [])

print(f"Query: {user_query}\n")
display(Markdown(answer))
if citations:
    print("\nCitations:")
    for i, c in enumerate(citations, 1):
        title = c.get("title", "n/a")
        search_score, rerank_score = score_map.get(title, (None, None))
        score_str = ""
        if rerank_score is not None:
            score_str = f"  (rerank: {rerank_score:.4f}, search: {search_score:.4f})"
        print(f"  [{i}] {title}{score_str}")

Query: Welche Tabelle enthält die Maximaltemperaturen für Routine Beförderungen?



Die Maximaltemperaturen für Routine-Beförderungsbedingungen sind in Tabelle 5 zusammengefasst, wie in den Dokumenten angegeben [doc1][doc2].


Citations:
  [1] 5.2.3 Berechnungsergebnisse für den Behälter und das Inventar  (rerank: 0.0164, search: 0.7173)
  [2] Tabellenverzeichnis  (rerank: 0.0161, search: 0.6450)
  [3] 1. Einleitung und Zusammenfassung  (rerank: 0.0159, search: 0.6213)
  [4] 6.1 Berechnungsmodelle  (rerank: 0.0156, search: 0.6064)
  [5] 3. Änderung (vgl. Kapitel 7.1):  (rerank: 0.0154, search: 0.5946)
